<a href="https://colab.research.google.com/github/ovifernandez/pruebaopengeoai/blob/develop/model-trainer-kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%capture
# 1. Instalamos geoai (al ser muy ligero, dejamos que lo baje de internet)
%pip install geoai-py -q


In [3]:
import os
import geoai

In [ ]:
train_raster_path = "/content/drive/MyDrive/AGRIA/input/parcelas/B7/B7.tif"
train_vector_path = "/content/drive/MyDrive/AGRIA/input/parcelas/B7/GroundTruth_B7.geojson"
test_raster_path = "/content/drive/MyDrive/AGRIA/input/parcelas/B9/20220714_FLEXIGROBOTS_B9_CIR.tif"


In [ ]:
geoai.get_raster_info(train_raster_path)

In [ ]:
style_dict = {
    "color": "#ff0000",
    "weight": 2,
    "opacity": 1,
    # "fill": True,
    # "fillColor": "#ffffff",
    "fillOpacity": 0,
    # "dashArray": "9"
    # "clickable": True,
}
style_function = lambda x: style_dict

geoai.view_vector_interactive(
    train_vector_path, tiles=train_raster_path, style_function=style_function
)

In [ ]:
geoai.view_raster(test_raster_path)

In [4]:
import os

input_folder = "/content/drive/MyDrive/AGRIA/input/parcelas/B7"
out_folder = "/content/drive/MyDrive/AGRIA/maskrcnn/mask_30epochs"
os.makedirs(out_folder, exist_ok=True)

Entrenamos el modelo Mask R-CNN sobre nuestros tiles generados, para que clasifique, localice bboxes y aplique máscaras a cada cepa detectada.

In [5]:
geoai.train_instance_segmentation_model(
    images_dir=f"{input_folder}/images",
    labels_dir=f"{input_folder}/labels",
    output_dir=f"{out_folder}/instance_models",
    num_classes=2,  # clase fondo y clase cepa. En un futuro, se añadirá clase tronco
    num_channels=3, # 3 para imágenes RGB, 5 para imágenes MSP
    batch_size=4, # Para no consumir excesiva VRAM, y no provocar un error de Out of Memory a mitad de ejecución.
    num_epochs=30, # 10 para una PoC, 50 para entrenamiento real con dataset augmentado.
    learning_rate=0.0005, # Learning rate menos agresivo que el de por defecto, para un descenso de gradiente suave y controlado con un batch sizze de 4.
    val_split=0.2,
    visualize=False,
    verbose=True,
)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:01<00:00, 171MB/s]


KeyboardInterrupt: 

In [ ]:
# Definimos las rutas a las máscaras predichas
masks_path = f"{out_folder}/test_instance_prediction.tif"
model_path = f"{out_folder}/instance_models/best_model.pth"

In [ ]:
geoai.view_raster(test_raster_path)

In [1]:
# Inferencia sobre nuevo ortomosaico, de la parcela B9
geoai.instance_segmentation(
    input_path=test_raster_path,
    output_path=masks_path,
    model_path=model_path,
    num_classes=2,
    num_channels=3,
    window_size=512,
    overlap=256,
    confidence_threshold=0.6,
    batch_size=4,
)

Mounted at /content/drive


In [ ]:
masks_path_high_conf = f"{out_folder}/test_instance_prediction_high_conf.tif"


In [ ]:
geoai.instance_segmentation(
    input_path=test_raster_path,
    output_path=masks_path_high_conf,
    model_path=model_path,
    num_classes=2,
    num_channels=3,
    window_size=512,
    overlap=256,
    confidence_threshold=0.7,  # Higher threshold for more confident predictions
    batch_size=4,
)

In [ ]:
output_vector_path = "test_instance_prediction.geojson"
gdf = geoai.orthogonalize(masks_path, output_vector_path, epsilon=2)